#### Tools Used

1. S3 Buckets
2. IAM Roles and Users
3. Complete Infrastructure of AWS Sagemaker


In [1]:
# %pip install numpy

In [2]:
# importing relevant libraries
import sagemaker
from sklearn.model_selection import train_test_split

import boto3
import pandas as pd

sagemaker.config INFO - Not applying SDK defaults from location: C:\ProgramData\sagemaker\sagemaker\config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: C:\Users\STUDSENT\AppData\Local\sagemaker\sagemaker\config.yaml


In [4]:
# connecting to sagemaker

sagemaker_boto3 = boto3.client("sagemaker")
session = sagemaker.Session()
region = session.boto_session.region_name

bucket="mob-price-sagemaker15"

print(bucket, region)

mob-price-sagemaker15 us-east-1


In [5]:
# reading the dataset
data = pd.read_csv("data/mob_price_classification_train.csv")
data.head(n=10)

,battery_power,blue,clock_speed,dual_sim,fc,four_g,int_memory,m_dep,mobile_wt,n_cores,...,px_height,px_width,ram,sc_h,sc_w,talk_time,three_g,touch_screen,wifi,price_range
0,842,0,2.2,0,1,0,7,0.6,188,2,...,20,756,2549,9,7,19,0,0,1,1
1,1021,1,0.5,1,0,1,53,0.7,136,3,...,905,1988,2631,17,3,7,1,1,0,2
2,563,1,0.5,1,2,1,41,0.9,145,5,...,1263,1716,2603,11,2,9,1,1,0,2
3,615,1,2.5,0,0,0,10,0.8,131,6,...,1216,1786,2769,16,8,11,1,0,0,2
4,1821,1,1.2,0,13,1,44,0.6,141,2,...,1208,1212,1411,8,2,15,1,1,0,1
5,1859,0,0.5,1,3,0,22,0.7,164,1,...,1004,1654,1067,17,1,10,1,0,0,1
6,1821,0,1.7,0,4,1,10,0.8,139,8,...,381,1018,3220,13,8,18,1,0,1,3
7,1954,0,0.5,1,0,0,24,0.8,187,4,...,512,1149,700,16,3,5,1,1,1,0
8,1445,1,0.5,0,0,0,53,0.7,174,7,...,386,836,1099,17,1,20,1,0,0,0
9,509,1,0.6,1,2,1,9,0.1,93,5,...,1137,1224,513,19,10,12,1,0,0,0


In [6]:
data.shape

(2000, 21)

In [7]:
# splitting the dataset
x = data.drop(columns=['price_range'])
y = data[['price_range']]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.15, random_state=42)

In [8]:
train_x = pd.DataFrame(x_train)
train_x['price_range'] = y_train

test_x = pd.DataFrame(x_test)
test_x['price_range'] = y_test

In [9]:
train_x.to_csv("train-v1.csv", index=False)
test_x.to_csv("test-v1.csv", index=False)

In [10]:
bucket

'mob-price-sagemaker15'

In [11]:
# sending the data to s3
sk_prefix = "sagemaker/mobile_price_classification/sklearncontainer"
train_path = session.upload_data(path='train-v1.csv', bucket=bucket, key_prefix=sk_prefix) 
test_path = session.upload_data(path='test-v1.csv', bucket=bucket, key_prefix=sk_prefix) 

print(train_path)
print(test_path)

s3://mob-price-sagemaker15/sagemaker/mobile_price_classification/sklearncontainer/train-v1.csv
s3://mob-price-sagemaker15/sagemaker/mobile_price_classification/sklearncontainer/test-v1.csv


In [12]:
import os
import io
import pandas as pd
import joblib

def model_fn(model_dir):
    """Load model from SageMaker model directory"""
    return joblib.load(os.path.join(model_dir, "model.joblib"))

def input_fn(request_body, request_content_type):
    """Parse incoming request"""
    if request_content_type == "text/csv":
        return pd.read_csv(io.StringIO(request_body))
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    """Run predictions"""
    return model.predict(input_data)

def output_fn(prediction, content_type):
    """Format predictions"""
    if content_type == "text/csv":
        out = io.StringIO()
        pd.DataFrame(prediction).to_csv(out, index=False, header=False)
        return out.getvalue()
    else:
        raise ValueError(f"Unsupported content type: {content_type}")


In [13]:
import os
import argparse
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report



if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--n_estimators", type=int, default=100)
    parser.add_argument("--random_state", type=int, default=42)

    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))


    parser.add_argument("--train-file", type=str, default="train-v1.csv")
    parser.add_argument("--test-file", type=str, default="test-v1.csv")
    args, _ = parser.parse_known_args()

    # args, _ = parser.parse_known_args()

    # Load data
    train_data = pd.read_csv(os.path.join(args.train, args.train_file))
    test_data = pd.read_csv(os.path.join(args.test, args.test_file))

    train_data = pd.read_csv(os.path.join(args.train, args.train_file))
    test_data = pd.read_csv(os.path.join(args.test, args.test_file))

    # Features & labels
    features = list(train_data.columns)
    label = features.pop(-1)
    X_train, y_train = train_data[features], train_data[label]
    X_test, y_test = test_data[features], test_data[label]

    # Train model
    model = RandomForestClassifier(
        n_estimators=args.n_estimators,
        random_state=args.random_state,
        verbose=2
    )
    model.fit(X_train, y_train)

    # Save model
    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))

    # Evaluate
    preds = model.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Classification Report:\n", classification_report(y_test, preds))


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:27                                                                                   │
│                                                                                                  │
│   24 │   # args, _ = parser.parse_known_args()                                                   │
│   25 │                                                                                           │
│   26 │   # Load data                                                                             │
│ ❱ 27 │   train_data = pd.read_csv(os.path.join(args.train, args.train_file))                     │
│   28 │   test_data = pd.read_csv(os.path.join(args.test, args.test_file))                        │
│   29 │                                                                                           │
│   30 │   train_data = pd.read_csv(os.path.join(args.train, args.train_file))                     │
│ in join:100                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
from sagemaker.sklearn.estimator import SKLearn

sklearn_estimator = SKLearn(
    entry_point="train.py",
    role="arn:aws:iam::314146298520:role/sagemaker-ml",
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="0.23-1",
    hyperparameters={"n_estimators": 100, "random_state": 42}
)

sklearn_estimator.fit({"train": train_path, "test": test_path})


: 

In [ ]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data=sklearn_estimator.model_data,
    role="arn:aws:iam::314146298520:role/sagemaker-ml",
    entry_point="inference.py",
    framework_version="0.23-1"
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge"
)


: 

In [ ]:
# %%writefile script.py

# # libraries
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score
# import sklearn
# import joblib
# import boto3
# import pathlib
# from io import  StringIO
# import argparse
# import os
# import numpy as np
# import pandas as pd


# # function to train the model
# def model_func(model_dir):
#     clf = joblib.load(os.path.join(model_dir, "model.joblib"))


# def predict_fn(input_data, model):
#     """Make predictions"""
#     return model.predict(input_data)


# if __name__ == '__main__':
#     print("[Info] Extracting Arguments")
#     parser = argparse.ArgumentParser()

#     # hyperparameters
#     parser.add_argument("--n_estimators", type=int, default=100)
#     parser.add_argument("--random_state", type=int, default=42)

#     # data, model and output directories
#     parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
#     parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
#     parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))


#     parser.add_argument("--train-file", type=str, default="train-v1.csv")
#     parser.add_argument("--test-file", type=str, default="test-v1.csv")

#     args, _ = parser.parse_known_args()

#     print("Sklearn Version: ", sklearn.__version__)
#     print("Joblib Version: ", joblib.__version__)

#     print("[INFO] Reading Data")
#     print()
#     train_data = pd.read_csv(os.path.join(args.train, args.train_file))
#     test_data = pd.read_csv(os.path.join(args.test, args.test_file))

#     # getting the features
#     features = list(train_data.columns)
#     label = features.pop(-1)

#     # getting the dataset
#     print("Building training and testing datasets")
#     print()
#     X_train = train_data[features]
#     X_test = test_data[features]
#     y_train = train_data[label]
#     y_test = test_data[label]

#     print('Column Order: ')
#     print(features)
#     print()

#     print("Label Column: ", label)
#     print()

#     print("Data Shape")
#     print()
#     print("--------- SHAPE OF TRAINING DATA (85%) ---------")
#     print(X_train.shape)
#     print(y_train.shape)
#     print()

#     print("--------- SHAPE OF TESTING DATA (15%) ---------")
#     print(X_test.shape)
#     print(y_test.shape)
#     print()

#     # building the model
#     print("Training RandomForest Model")
#     print()

#     model = RandomForestClassifier(
#         n_estimators=args.n_estimators,
#         random_state=args.random_state,
#         verbose=2, 
#         n_jobs=1
#     )

#     model.fit(X_train, y_train)

#     print()

#     model_path = os.path.join(args.model_dir, "model.joblib")
#     joblib.dump(model, model_path)

#     print(f"Model Saved at {model_path}")

#     # evaluating the model
#     y_prep_test = model.predict(X_test)
#     test_accuracy = accuracy_score(y_test, y_prep_test)
#     test_report = classification_report(y_test, y_prep_test)

#     print()
#     print("--------- METRICS RESULT FOR TESTING DATA ---------")
#     print()
#     print("Total Rows are ", X_test.shape[0])
#     print('[TESTING] Model Accuracy is ', test_accuracy)
#     print('[TESTING] Testing Report')
#     print(test_report)

Overwriting script.py


: 

In [ ]:
# # AWS Sagemaker Entry Point for Model Training

# from sagemaker.sklearn.estimator import SKLearn

# FRAMEWORK_VERSION="0.23-1"

# sklearn_estimator = SKLearn(
#     entry_point="script.py",
#     role='arn:aws:iam::314146298520:role/sagemaker-ml',
#     instance_count=1,
#     instance_type='ml.m5.large',
#     framework_version=FRAMEWORK_VERSION,
#     base_job_name="Simple-RF-Classifier", 
#     hyperparameters={
#         "n_estimators": 100,
#         "random_state": 42
#     },
#     use_spot_instance=True,
#     max_run=3600
# )

: 

In [ ]:
# from time import gmtime, strftime 

# sklearn_estimator.latest_training_job.wait(logs="None")
# artifact = sagemaker_boto3.describe_training_job(
#     TrainingJobName=sklearn_estimator.latest_training_job.name
# )["ModelArtifacts"]["S3ModelArtifacts"]

: 

In [ ]:
# # endpoint deployment

# endpoint_name = 'Custom-sklearn-model-' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
# print(f"EndpointName={endpoint_name}")

# predictor=model.deploy(
#     initial_instance_count=1,
#     instance_type="ml.t2.medium",
#     endpoint_name=endpoint_name
# )

: 

: 

In [ ]:
# launching the training job
# sklearn_estimator.fit({"train": train_path, "test": test_path}, wait=True)

INFO:sagemaker:Creating training-job with name: Simple-RF-Classifier-2025-08-13-17-45-11-543


2025-08-13 17:45:25 Starting - Starting the training job...
2025-08-13 17:46:03 Downloading - Downloading input data...
2025-08-13 17:46:28 Downloading - Downloading the training image...
2025-08-13 17:47:19 Training - Training image download completed. Training in progress.
2025-08-13 17:47:19 Uploading - Uploading generated training model2025-08-13 17:47:12,952 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2025-08-13 17:47:12,956 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-08-13 17:47:13,000 sagemaker_sklearn_container.training INFO     Invoking user training script.
2025-08-13 17:47:13,280 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-08-13 17:47:13,292 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-08-13 17:47:13,305 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-08-13 17:47:13

: 

In [ ]:
# 

# sklearn_estimator.latest_training_job.wait(logs="None")
# artifact = sagemaker_boto3.describe_training_job(
#     TrainingJobName=sklearn_estimator.latest_training_job.name
# )["ModelArtifacts"]["S3ModelArtifacts"]


2025-08-13 17:47:37 Starting - Preparing the instances for training
2025-08-13 17:47:37 Downloading - Downloading the training image
2025-08-13 17:47:37 Training - Training image download completed. Training in progress.
2025-08-13 17:47:37 Uploading - Uploading generated training model
2025-08-13 17:47:37 Completed - Training job completed


: 

In [ ]:
artifact

's3://sagemaker-us-east-1-314146298520/Simple-RF-Classifier-2025-08-13-17-45-11-543/output/model.tar.gz'

: 

### Model Deployment with Endpoints

In [ ]:
# from sagemaker.sklearn.model import SKLearnModel
# from time import gmtime, strftime

# model_name = 'Custom-sklearn-model-' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
# model = SKLearnModel(
#     name=model_name,
#     model_data=artifact,
#     role="arn:aws:iam::314146298520:role/sagemaker-ml",
#     entry_point="script.py",
#     framework_version=FRAMEWORK_VERSION
# )


: 

In [ ]:
# model

: 

In [ ]:
# # endpoint deployment

# endpoint_name = 'Custom-sklearn-model-' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
# print(f"EndpointName={endpoint_name}")

# predictor=model.deploy(
#     initial_instance_count=1,
#     instance_type="ml.t2.medium",
#     endpoint_name=endpoint_name
# )

EndpointName=Custom-sklearn-model-2025-08-13-18-34-09


INFO:sagemaker:Creating model with name: Custom-sklearn-model-2025-08-13-17-48-33
INFO:sagemaker:Creating endpoint-config with name Custom-sklearn-model-2025-08-13-18-34-09
INFO:sagemaker:Creating endpoint with name Custom-sklearn-model-2025-08-13-18-34-09


-----------------------------------------------------*

ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-endpoint


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:6                                                                                    │
│                                                                                                  │
│    3 endpoint_name = 'Custom-sklearn-model-' + strftime("%Y-%m-%d-%H-%M-%S", gmtime())           │
│    4 print(f"EndpointName={endpoint_name}")                                                      │
│    5                                                                                             │
│ ❱  6 predictor=model.deploy(                                                                     │
│    7 │   initial_instance_count=1,                                                               │
│    8 │   instance_type="ml.t2.medium",                                                           │
│    9 │   endpoint_name=endpoint_name                                                             │
│                                                                                                  │
│ c:\Users\STUDSENT\Desktop\machine-learning-Ops\aws_sagemaker_MLOps\venv\Lib\site-packages\sagema │
│ ker\model.py:1814 in deploy                                                                      │
│                                                                                                  │
│   1811 │   │   │   │   )                                                                         │
│   1812 │   │   │   │   self.sagemaker_session.update_endpoint(self.endpoint_name, endpoint_conf  │
│   1813 │   │   │   else:                                                                         │
│ ❱ 1814 │   │   │   │   self.sagemaker_session.endpoint_from_production_variants(                 │
│   1815 │   │   │   │   │   name=self.endpoint_name,                                              │
│   1816 │   │   │   │   │   production_variants=[production_variant],                             │
│   1817 │   │   │   │   │   tags=tags,                                                            │
│                                                                                                  │
│ c:\Users\STUDSENT\Desktop\machine-learning-Ops\aws_sagemaker_MLOps\venv\Lib\site-packages\sagema │
│ ker\session.py:6250 in endpoint_from_production_variants                                         │
│                                                                                                  │
│   6247 │   │   logger.info("Creating endpoint-config with name %s", name)                        │
│   6248 │   │   self.sagemaker_client.create_endpoint_config(**config_options)                    │
│   6249 │   │                                                                                     │
│ ❱ 6250 │   │   return self.create_endpoint(                                                      │
│   6251 │   │   │   endpoint_name=name,                                                           │
│   6252 │   │   │   config_name=name,                                                             │
│   6253 │   │   │   tags=endpoint_tags,                                                           │
│                                                                                                  │
│ c:\Users\STUDSENT\Desktop\machine-learning-Ops\aws_sagemaker_MLOps\venv\Lib\site-packages\sagema │
│ ker\session.py:5095 in create_endpoint                                                           │
│                                                                                                  │
│   5092 │   │   │   logger.error(                                                                 │
│   5093 │   │   │   │   "Please check the troubleshooting guide for common errors: %s", troubles  │
│   5094 │   │   │   )                                                                             │
│ ❱ 5095 │   │   │   raise e                                 

: 

In [ ]:
# predictor.predict(X_test[features][:2].values.tolist())

: 